# PubMed pathogen category assignment

This notebook reads the accepted abstracts from notebook 01, classifies
abstract-level animal-infection and zoonosis evidence, and derives the
corpus-relative categories 1, 2, and 3. It writes a pathogen-level category
table under outputs/pubmed_screening/.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    derive_category_counts,
    load_corpus_articles,
    screen_pubmed_corpus,
)

## Configure the refined corpus, pathogen aliases, and LLM

In [ ]:
REFINED_CORPUS_PATH = PROJECT_ROOT / 'outputs' / 'pubmed_screening' / 'refined_corpus_articles.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'pubmed_screening'
CORPUS_PATHOGENS = None  # e.g. ['Nipah virus']
CORPUS_START_YEAR = None
CORPUS_END_YEAR = None

PATHOGENS = {
    'Nipah virus': ['Nipah', 'NiV'],
}
OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
LLM_MAX_TOKENS = 1024
LLM_RETRIES = 3
LLM_MAX_CALLS = None
LLM_SAVE_EVERY = 10
RESUME = True
RETRY_FAILED_LLM_ROWS = False


## Load and filter the accepted corpus

In [ ]:
corpus = load_corpus_articles(
    REFINED_CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
print('Accepted corpus rows:', len(corpus))
if corpus.empty:
    raise ValueError('No accepted abstracts remain after category-stage filtering.')
display(corpus.groupby('pathogen').size().rename('abstracts').reset_index())

## Classify animal-infection and zoonosis evidence

In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the category screen.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
screening_run = screen_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Category decisions:', len(screening_run.screening))
print('Classification failures:', len(screening_run.failures))

## Derive categories and associate them with pathogens

In [ ]:
screened_with_categories, zoonosis_timeline, category_counts = derive_category_counts(
    screening_run.screening,
    OUTPUT_DIR,
)
pathogen_category_table = pd.read_parquet(OUTPUT_DIR / 'pathogen_category_table.parquet')
print('First confirmed zoonosis by pathogen:')
display(zoonosis_timeline)
print('Pathogen category table:')
display(pathogen_category_table)
print('Yearly and total category counts:')
display(category_counts.sort_values(['pathogen', 'publication_year', 'category'], na_position='first'))

Category 1 means no confirmed zoonosis evidence was found in the searched
corpus for that pathogen; it is not proof that the pathogen has never
undergone zoonosis. Review-required and failed rows are excluded from final
counts. The category table is a corpus-level summary, not a biological truth
label.

In [ ]:
print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
print('Review-required rows:', int(screened_with_categories['review_required'].fillna(True).sum()))
display(screened_with_categories[[
    'pathogen', 'pmid', 'llm_category', 'final_category',
    'target_pathogen_supported', 'animal_infection_supported',
    'zoonosis_supported', 'confidence', 'review_required', 'rationale'
]].head(20))